# Import Required Libraries

In [24]:
import torch
from transformers import LlavaNextForConditionalGeneration, LlavaNextProcessor, AutoConfig
from PIL import Image
import requests
from io import BytesIO
import os

# Load Model and Processor (Run Once)

In [ ]:
# 清理 GPU 缓存，避免之前的错误状态影响加载
import torch
torch.cuda.empty_cache()

# 重置 CUDA 错误状态
if torch.cuda.is_available():
    try:
        torch.cuda.synchronize()
    except:
        pass
    
print("GPU cache cleared.")

In [25]:
# 模型路径
model_path = "/root/workspace/models/llama3-llava-next-8b-hf"

print(f"Loading model from {model_path}...")

# 加载 Processor
processor = LlavaNextProcessor.from_pretrained(model_path)
print("Processor loaded.")

# 加载模型
model = LlavaNextForConditionalGeneration.from_pretrained(
    model_path,
    dtype=torch.float16,
    device_map="auto",
)

print("Model loaded successfully.")

Loading model from /root/workspace/models/llama3-llava-next-8b-hf...
Processor loaded.
Processor loaded.


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


# Define Helper Functions

In [ ]:
def load_image(image_path):
    if image_path.startswith('http') or image_path.startswith('https'):
        response = requests.get(image_path)
        image = Image.open(BytesIO(response.content)).convert('RGB')
    else:
        image = Image.open(image_path).convert('RGB')
    return image

# Configure Inputs

In [ ]:
# 在这里修改输入，然后运行下一个单元格进行测试
text_input = "Describe this image in detail."

# 图片路径或URL (留空则仅进行文本对话)
# image_path = "https://llava-vl.github.io/static/images/view.jpg" 
image_path = "/root/workspace/align/test_img.png"

# Run Inference

In [ ]:
try:
    # 加载图片
    image = None
    if image_path:
        image = load_image(image_path)
        print(f"✅ Image loaded from {image_path}")

    # 构建 Conversation
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": text_input},
            ],
        },
    ]
    
    if image:
        conversation[0]["content"].append({"type": "image"})

    # 构建 Prompt
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    
    # 处理输入
    if image:
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    else:
        inputs = processor(text=prompt, return_tensors="pt").to(model.device)

    # 生成回复
    print("🤖 AI Generating...")
    
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
        )

    # 解码输出
    print("\n=== Response ===")
    print(processor.decode(output[0], skip_special_tokens=True))

except Exception as e:
    print(f"An error occurred: {e}")
    import traceback
    traceback.print_exc()
finally:
    torch.cuda.empty_cache()